In [ ]:
#Configuration de l'environnement
# Initialisation de la session Snowflake et import de toutes les bibliothèques nécessaires au pipeline ML.

from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
from scipy.stats import randint

# Récupération de la session active Snowflake
session = get_active_session()

print("✅ Session Snowflake initialisée")
print(f"   Rôle     : {session.get_current_role()}")
print(f"   Database : {session.get_current_database()}")
print(f"   Schema   : {session.get_current_schema()}")

## ÉTAPE 1 — Configuration de l'environnement

In [ ]:
CREATE DATABASE IF NOT EXISTS HOUSE_PRICE_DB;
CREATE SCHEMA IF NOT EXISTS HOUSE_PRICE_DB.ML;
CREATE WAREHOUSE IF NOT EXISTS ML_WH
  WAREHOUSE_SIZE = 'MEDIUM'
  AUTO_SUSPEND = 300
  AUTO_RESUME = TRUE;

USE DATABASE HOUSE_PRICE_DB;
USE SCHEMA ML;
USE WAREHOUSE ML_WH;

In [ ]:
import snowflake.snowpark as snowpark
from snowflake.snowpark.session import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')


## ÉTAPE 2 — Ingestion des données depuis S3

In [ ]:
-- Créer le stage externe pointant vers S3
CREATE OR REPLACE STAGE HOUSE_PRICE_STAGE
  URL = 's3://logbrain-datalake/datasets/house_price/'
  FILE_FORMAT = (
    TYPE = 'CSV'
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    SKIP_HEADER = 1
    NULL_IF = ('NULL', 'null', '')
    EMPTY_FIELD_AS_NULL = TRUE
  );

-- Vérifier que les fichiers sont visibles
LIST @HOUSE_PRICE_STAGE;

In [ ]:
TRUNCATE TABLE HOUSE_PRICES_RAW;

COPY INTO HOUSE_PRICES_RAW
FROM (
  SELECT
    $1:price::NUMBER,
    $1:area::NUMBER,
    $1:bedrooms::NUMBER,
    $1:bathrooms::NUMBER,
    $1:stories::NUMBER,
    $1:mainroad::VARCHAR,
    $1:guestroom::VARCHAR,
    $1:basement::VARCHAR,
    $1:hotwaterheating::VARCHAR,
    $1:airconditioning::VARCHAR,
    $1:parking::NUMBER,
    $1:prefarea::VARCHAR,
    $1:furnishingstatus::VARCHAR
  FROM @HOUSE_PRICE_STAGE/Housing_Price_Data.json
)
FILE_FORMAT = (TYPE = 'JSON' STRIP_OUTER_ARRAY = TRUE)
ON_ERROR = 'CONTINUE';

-- Vérification
SELECT COUNT(*) AS nb_lignes FROM HOUSE_PRICES_RAW;
SELECT * FROM HOUSE_PRICES_RAW LIMIT 5;

## ÉTAPE 3 — Exploration des données

In [ ]:
# Charger depuis la table Snowflake
df_sp = session.table("HOUSE_PRICES_RAW")
df = df_sp.to_pandas()

print(f"✅ {len(df)} lignes chargées")
print(f"Colonnes : {list(df.columns)}")
print(df.head(3))

In [ ]:
# Statistiques descriptives
print("=== Statistiques descriptives ===")
display(df.describe())

print("\n=== Valeurs manquantes ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "✅ Aucune valeur manquante")

print("\n=== Types des colonnes ===")
print(df.dtypes)

In [ ]:
# Distribution du prix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['PRICE'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution du prix')
axes[0].set_xlabel('Prix')
axes[1].hist(np.log1p(df['PRICE']), bins=30, color='coral', edgecolor='white')
axes[1].set_title('Distribution log(prix)')
axes[1].set_xlabel('log(Prix)')
plt.tight_layout()
plt.show()

In [ ]:
# Matrice de corrélation
numerical_cols = ['PRICE','AREA','BEDROOMS','BATHROOMS','STORIES','PARKING']
corr_matrix = df[numerical_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Matrice de corrélation — variables numériques')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots catégorielles
cat_cols = ['MAINROAD','GUESTROOM','BASEMENT','HOTWATERHEATING',
            'AIRCONDITIONING','PREFAREA','FURNISHINGSTATUS']

# Normaliser en minuscules
df_plot = df.copy()
for col in cat_cols:
    df_plot[col] = df_plot[col].astype(str).str.lower().str.strip()

print("Valeurs uniques par colonne :")
for col in cat_cols:
    print(f"  {col}: {df_plot[col].unique()}")

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    groups = df_plot.groupby(col)['PRICE'].apply(list)
    axes[i].boxplot(list(groups.values), labels=list(groups.index), patch_artist=True)
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=15)
axes[-1].set_visible(False)
plt.suptitle('Prix selon les variables catégorielles')
plt.tight_layout()
plt.show()

In [ ]:
#  Encodage et préparation
binary_cols = ['MAINROAD','GUESTROOM','BASEMENT','HOTWATERHEATING',
               'AIRCONDITIONING','PREFAREA']

df_clean = df.copy()

# Normaliser en minuscules avant encodage
for col in binary_cols + ['FURNISHINGSTATUS']:
    df_clean[col] = df_clean[col].astype(str).str.lower().str.strip()

# Encodage yes/no → 1/0
for col in binary_cols:
    df_clean[col] = df_clean[col].map({'yes': 1, 'no': 0})

# Encodage ordinal ameublement
furnishing_map = {'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0}
df_clean['FURNISHINGSTATUS'] = df_clean['FURNISHINGSTATUS'].map(furnishing_map)

print("Valeurs NaN après encodage :")
print(df_clean.isnull().sum())
print(f"\nLignes restantes : {len(df_clean)}")

# Séparation features / cible
X = df_clean.drop(columns=['PRICE'])
y = df_clean['PRICE']

# Normalisation
scaler = StandardScaler()
num_cols = ['AREA','BEDROOMS','BATHROOMS','STORIES','PARKING']
X_scaled = X.copy()
X_scaled[num_cols] = scaler.fit_transform(X[num_cols])

# Split train / test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
print(f"\n✅ Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")

# Sauvegarder dans Snowflake
df_features = df_clean.copy()
df_features['SPLIT'] = 'train'
df_features.loc[X_test.index, 'SPLIT'] = 'test'
session.create_dataframe(df_features).write.mode("overwrite").save_as_table("HOUSE_PRICES_FEATURES")
print("✅ Table HOUSE_PRICES_FEATURES sauvegardée")

## ÉTAPE 4 — Préparation des features

In [ ]:
# Séparation des features (X) et de la variable cible (y)

# On retire la colonne PRICE qui est notre variable à prédire
X = df_clean.drop(columns=['PRICE'])
y = df_clean['PRICE']

print(f"✅ Features X : {X.shape[0]} lignes × {X.shape[1]} colonnes")
print(f"   Colonnes   : {list(X.columns)}")
print(f"\n✅ Cible y    : {y.shape[0]} valeurs")
print(f"   Min prix   : {y.min():,.0f}")
print(f"   Max prix   : {y.max():,.0f}")
print(f"   Moy prix   : {y.mean():,.0f}")

In [ ]:
# Normalisation des variables numériques
# On applique un StandardScaler pour centrer-réduire les features numériques (moyenne=0, écart-type=1).

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Colonnes numériques à normaliser
num_cols = ['AREA', 'BEDROOMS', 'BATHROOMS', 'STORIES', 'PARKING']

X_scaled = X.copy()
X_scaled[num_cols] = scaler.fit_transform(X[num_cols])

# Vérification : après scaling, moyenne ≈ 0 et écart-type ≈ 1
print("✅ Vérification après normalisation :")
print(X_scaled[num_cols].describe().round(3))

In [ ]:
#Séparation train / test
# On réserve 20% des données pour l'évaluation finale.

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42
)

print(f"✅ Données d'entraînement : {X_train.shape[0]} lignes ({80}%)")
print(f"✅ Données de test        : {X_test.shape[0]} lignes ({20}%)")
print(f"\n   Répartition vérifiée  : {X_train.shape[0] + X_test.shape[0]} lignes au total")

In [ ]:
# Sauvegarde de la table des features dans Snowflake
# On ajoute une colonne SPLIT pour distinguer les lignes d'entraînement et de test, utile pour les étapes suivantes.

# Ajouter la colonne SPLIT au dataset nettoyé
df_features = df_clean.copy()
df_features['SPLIT'] = 'train'
df_features.loc[X_test.index, 'SPLIT'] = 'test'

# Écriture dans Snowflake
session.create_dataframe(df_features) \
       .write.mode("overwrite") \
       .save_as_table("HOUSE_PRICES_FEATURES")

# Vérification depuis Snowflake
count = session.table("HOUSE_PRICES_FEATURES").count()
splits = session.table("HOUSE_PRICES_FEATURES") \
                .group_by("SPLIT") \
                .count() \
                .to_pandas()

print(f"✅ Table HOUSE_PRICES_FEATURES sauvegardée ({count} lignes)")
print("\n   Répartition train/test :")
print(splits.to_string(index=False))

## ÉTAPE 5 — Entraînement des modèles

In [ ]:
# Import de toutes les bibliothèques nécessaires pour l'entraînement et l'évaluation des modèles ML.

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import time

print("✅ Bibliothèques ML importées avec succès")

In [ ]:
# Entraînement et comparaison de plusieurs modèles
# Test de 5 algorithmes différents pour identifier le plus performant sur notre problème de régression immobilière.

# Dictionnaire des modèles à comparer
models = {
    "Linear Regression"    : LinearRegression(),
    "Ridge"                : Ridge(alpha=1.0),
    "Lasso"                : Lasso(alpha=1.0),
    "Random Forest"        : RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost"              : xgb.XGBRegressor(n_estimators=100, learning_rate=0.1,
                                               random_state=42, verbosity=0),
}

# Entraînement et évaluation de chaque modèle
results = {}
print(f"{'Modèle':<25} {'RMSE':>15} {'MAE':>15} {'R²':>8}")
print("─" * 68)

for name, model in models.items():
    # Entraînement
    start = time.time()
    model.fit(X_train, y_train)
    duration = time.time() - start

    # Prédictions sur le jeu de test
    y_pred = model.predict(X_test)

    # Calcul des métriques
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    # Stockage des résultats
    results[name] = {
        "model"   : model,
        "y_pred"  : y_pred,
        "RMSE"    : rmse,
        "MAE"     : mae,
        "R²"      : r2,
        "temps_s" : round(duration, 2)
    }

    print(f"{name:<25} {rmse:>15,.0f} {mae:>15,.0f} {r2:>8.4f}  ({duration:.2f}s)")

In [ ]:
# Visualisation des performances
# Comparaison graphique des métriques pour identifier visuellement le meilleur modèle de base.

# Tableau de synthèse trié par R² décroissant
results_df = pd.DataFrame([
    {"Modèle": k, "RMSE": v["RMSE"], "MAE": v["MAE"], "R²": v["R²"], "Temps (s)": v["temps_s"]}
    for k, v in results.items()
]).sort_values("R²", ascending=False).reset_index(drop=True)

print("=== Classement des modèles ===")
display(results_df)

# Graphique comparatif des 3 métriques
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics   = ['RMSE', 'MAE', 'R²']
colors    = ['steelblue', 'coral', 'seagreen']

for ax, metric, color in zip(axes, metrics, colors):
    bars = ax.barh(results_df['Modèle'], results_df[metric],
                   color=color, alpha=0.85, edgecolor='white')
    ax.set_title(f'Comparaison — {metric}', fontweight='bold')
    ax.set_xlabel(metric)
    # Afficher les valeurs sur les barres
    for bar, val in zip(bars, results_df[metric]):
        label = f'{val:.4f}' if metric == 'R²' else f'{val:,.0f}'
        ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
                label, va='center', fontsize=9)

plt.suptitle('Étape 5 — Comparaison des modèles de base', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Analyse visuelle du meilleur modèle

# Sélection automatique du meilleur modèle (R² le plus élevé)
best_name  = results_df.iloc[0]['Modèle']
best       = results[best_name]

print(f"🏆 Meilleur modèle de base : {best_name}")
print(f"   R²   = {best['R²']:.4f}")
print(f"   RMSE = {best['RMSE']:,.0f}")
print(f"   MAE  = {best['MAE']:,.0f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Prédictions vs Réalité
axes[0].scatter(y_test, best['y_pred'], alpha=0.5,
                color='steelblue', edgecolors='white', s=40)
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()], 'r--', lw=2, label='Parfait')
axes[0].set_xlabel('Prix réel')
axes[0].set_ylabel('Prix prédit')
axes[0].set_title(f'Prédictions vs Réalité\n{best_name}')
axes[0].legend()

# Graphique 2 : Distribution des résidus
residuals = y_test - best['y_pred']
axes[1].hist(residuals, bins=30, color='coral', edgecolor='white')
axes[1].axvline(x=0, color='black', linestyle='--', lw=2)
axes[1].set_xlabel('Résidu (réel − prédit)')
axes[1].set_ylabel('Fréquence')
axes[1].set_title(f'Distribution des résidus\n{best_name}')

plt.suptitle(f'Étape 5 — Analyse du meilleur modèle : {best_name}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Etape 6 — Évaluation approfondie

In [ ]:
# Évaluation du modèle
# On approfondit l'analyse des performances pour identifier les forces et limites de chaque modèle entraîné.

# Métriques enrichies avec MAPE (erreur % moyenne)
def mape(y_true, y_pred):
    """Mean Absolute Percentage Error — erreur relative moyenne en %"""
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f"{'Modèle':<25} {'RMSE':>12} {'MAE':>12} {'R²':>8} {'MAPE':>8}")
print("─" * 72)

eval_results = []
for name, v in results.items():
    mape_val = mape(y_test.values, v['y_pred'])
    eval_results.append({
        "Modèle"   : name,
        "RMSE"     : v['RMSE'],
        "MAE"      : v['MAE'],
        "R²"       : v['R²'],
        "MAPE (%)" : round(mape_val, 2)
    })
    print(f"{name:<25} {v['RMSE']:>12,.0f} {v['MAE']:>12,.0f} "
          f"{v['R²']:>8.4f} {mape_val:>7.2f}%")

eval_df = pd.DataFrame(eval_results).sort_values("R²", ascending=False).reset_index(drop=True)
print("\n=== Classement final ===")
display(eval_df)

In [ ]:
# Importance des variables
# XGBoost calcule l'importance de chaque feature dans la construction des arbres. Cela permet de comprendre quels
# facteurs influencent le plus le prix d'une maison.

xgb_model = results['XGBoost']['model']

# Récupérer les importances
feature_importance = pd.DataFrame({
    'Feature'    : X_train.columns,
    'Importance' : xgb_model.feature_importances_
}).sort_values('Importance', ascending=True)

# Graphique
plt.figure(figsize=(9, 6))
bars = plt.barh(feature_importance['Feature'], feature_importance['Importance'],
                color='steelblue', alpha=0.85, edgecolor='white')
plt.xlabel('Importance (gain)')
plt.title('Étape 6 — Importance des features — XGBoost', fontweight='bold')

# Valeurs sur les barres
for bar, val in zip(bars, feature_importance['Importance']):
    plt.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
             f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n📊 Top 3 features les plus influentes :")
top3 = feature_importance.sort_values('Importance', ascending=False).head(3)
for i, row in top3.iterrows():
    print(f"   {row['Feature']:<20} → {row['Importance']:.4f}")

In [ ]:
# Analyse des résidus du meilleur modèle
# Un bon modèle doit avoir des résidus :
#   - centrés autour de 0 (pas de biais systématique)
#   - distribués de façon homogène (homoscédasticité)
#   - sans structure visible (aléatoires)

y_pred_xgb = results['XGBoost']['y_pred']
residuals   = y_test.values - y_pred_xgb

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Graphique 1 : Résidus vs Prédictions
axes[0].scatter(y_pred_xgb, residuals, alpha=0.5, color='steelblue', edgecolors='white', s=40)
axes[0].axhline(y=0, color='red', linestyle='--', lw=2)
axes[0].set_xlabel('Prix prédit')
axes[0].set_ylabel('Résidu')
axes[0].set_title('Résidus vs Prédictions')

# Graphique 2 : Distribution des résidus
axes[1].hist(residuals, bins=30, color='coral', edgecolor='white', alpha=0.85)
axes[1].axvline(x=0, color='black', linestyle='--', lw=2)
axes[1].axvline(x=residuals.mean(), color='red', linestyle='-', lw=2,
                label=f'Moyenne = {residuals.mean():,.0f}')
axes[1].set_xlabel('Résidu')
axes[1].set_ylabel('Fréquence')
axes[1].set_title('Distribution des résidus')
axes[1].legend()

# Graphique 3 : Prédictions vs Réalité avec bande d'erreur ±10%
axes[2].scatter(y_test, y_pred_xgb, alpha=0.5, color='seagreen', edgecolors='white', s=40)
axes[2].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()], 'r--', lw=2, label='Parfait')
axes[2].fill_between([y_test.min(), y_test.max()],
                     [y_test.min()*0.9, y_test.max()*0.9],
                     [y_test.min()*1.1, y_test.max()*1.1],
                     alpha=0.15, color='red', label='±10%')
axes[2].set_xlabel('Prix réel')
axes[2].set_ylabel('Prix prédit')
axes[2].set_title('Prédictions vs Réalité (±10%)')
axes[2].legend()

plt.suptitle('Étape 6 — Analyse approfondie des résidus — XGBoost',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Résumé statistique des résidus
print("📊 Statistiques des résidus :")
print(f"   Moyenne  : {residuals.mean():>12,.0f}  (idéal = 0)")
print(f"   Std      : {residuals.std():>12,.0f}")
print(f"   Min      : {residuals.min():>12,.0f}")
print(f"   Max      : {residuals.max():>12,.0f}")
print(f"\n   % de prédictions dans ±10% du prix réel : "
      f"{(np.abs(residuals/y_test.values) < 0.10).mean()*100:.1f}%")
print(f"   % de prédictions dans ±20% du prix réel : "
      f"{(np.abs(residuals/y_test.values) < 0.20).mean()*100:.1f}%")

## Etape 7 - optimisation des hyperparamètres

In [ ]:
# Grid Search XGBoost (adapté Snowflake)


from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import randint

# Grille réduite pour Snowflake (évite les crashes mémoire)
param_grid_xgb = {
    'n_estimators'  : [100, 200],
    'max_depth'     : [3, 5],
    'learning_rate' : [0.05, 0.1],
    'subsample'     : [0.8, 1.0],
}

print("🔍 Lancement du Grid Search XGBoost...")
print(f"   Combinaisons : {2*2*2*2} × 5 folds = {2*2*2*2*5} fits")
print(f"   Mode         : séquentiel (n_jobs=1 — requis sur Snowflake)\n")

xgb_base = xgb.XGBRegressor(random_state=42, verbosity=0)

grid_search = GridSearchCV(
    xgb_base,
    param_grid_xgb,
    cv=5,
    scoring='r2',
    n_jobs=1,       
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f"\n✅ Grid Search terminé !")
print(f"   Meilleurs paramètres : {grid_search.best_params_}")
print(f"   Meilleur score CV R² : {grid_search.best_score_:.4f}")

# Évaluation sur le jeu de test
y_pred_grid = grid_search.best_estimator_.predict(X_test)
rmse_grid   = np.sqrt(mean_squared_error(y_test, y_pred_grid))
mae_grid    = mean_absolute_error(y_test, y_pred_grid)
r2_grid     = r2_score(y_test, y_pred_grid)

print(f"\n📊 Performance sur le jeu de test :")
print(f"   R²   = {r2_grid:.4f}  (base : 0.9036)")
print(f"   RMSE = {rmse_grid:,.0f}  (base : 29,314)")
print(f"   MAE  = {mae_grid:,.0f}  (base : 18,497)")

In [ ]:
# Random Search Random Forest

param_dist_rf = {
    'n_estimators'      : randint(50, 300),
    'max_depth'         : randint(3, 15),
    'min_samples_split' : randint(2, 10),
    'min_samples_leaf'  : randint(1, 5),
    'max_features'      : ['sqrt', 'log2'],
}

print("🔍 Lancement du Random Search Random Forest...")
print(f"   30 itérations × 5 folds = 150 fits")
print(f"   Mode : séquentiel (n_jobs=1)\n")

rf_base = RandomForestRegressor(random_state=42)

random_search = RandomizedSearchCV(
    rf_base,
    param_dist_rf,
    n_iter=30,       
    cv=5,
    scoring='r2',
    n_jobs=1,        
    random_state=42,
    verbose=1
)
random_search.fit(X_train, y_train)

print(f"\n✅ Random Search terminé !")
print(f"   Meilleurs paramètres : {random_search.best_params_}")
print(f"   Meilleur score CV R² : {random_search.best_score_:.4f}")

y_pred_rf = random_search.best_estimator_.predict(X_test)
rmse_rf   = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf    = mean_absolute_error(y_test, y_pred_rf)
r2_rf     = r2_score(y_test, y_pred_rf)

print(f"\n📊 Performance sur le jeu de test :")
print(f"   R²   = {r2_rf:.4f}")
print(f"   RMSE = {rmse_rf:,.0f}")
print(f"   MAE  = {mae_rf:,.0f}")

In [ ]:
# Sélection du meilleur modèle final
# On compare tous les modèles (base + optimisés) et on sélectionne celui avec le meilleur R² sur le jeu de test.

# Ajout des modèles optimisés au dictionnaire des résultats
results['XGBoost Optimisé (Grid)'] = {
    'model'  : grid_search.best_estimator_,
    'y_pred' : y_pred_grid,
    'RMSE'   : rmse_grid,
    'MAE'    : mae_grid,
    'R²'     : r2_grid
}
results['Random Forest Optimisé (Random)'] = {
    'model'  : random_search.best_estimator_,
    'y_pred' : y_pred_rf,
    'RMSE'   : rmse_rf,
    'MAE'    : mae_rf,
    'R²'     : r2_rf
}

# Tableau comparatif complet — tous modèles
final_df = pd.DataFrame([
    {"Modèle": k, "RMSE": v["RMSE"], "MAE": v["MAE"], "R²": v["R²"]}
    for k, v in results.items()
]).sort_values("R²", ascending=False).reset_index(drop=True)

print("=== Classement final — tous modèles ===")
display(final_df)

# Graphique comparatif R²
plt.figure(figsize=(10, 5))
bars = plt.barh(
    final_df['Modèle'],
    final_df['R²'],
    color=['gold' if i == 0 else 'steelblue' for i in range(len(final_df))],
    alpha=0.85,
    edgecolor='white'
)
plt.axvline(x=0.9036, color='red', linestyle='--', lw=1.5, label='Base XGBoost (0.9036)')
plt.xlabel('R²')
plt.title('Étape 7 — Comparaison finale R² — Base vs Optimisés', fontweight='bold')
plt.legend()
for bar, val in zip(bars, final_df['R²']):
    plt.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
             f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

# Sélection automatique du champion
best_final_name  = final_df.iloc[0]['Modèle']
best_final_model = results[best_final_name]['model']

print(f"\n🏆 Modèle sélectionné pour la production : {best_final_name}")
print(f"   R²   = {final_df.iloc[0]['R²']:.4f}")
print(f"   RMSE = {final_df.iloc[0]['RMSE']:,.0f}")
print(f"   MAE  = {final_df.iloc[0]['MAE']:,.0f}")

## Etape 8 — Model Registry

In [ ]:
# Stockage du meilleur modèle dans le Model Registry

from snowflake.ml.registry import Registry

# Initialisation du registry dans notre database/schema
reg = Registry(
    session=session,
    database_name="HOUSE_PRICE_DB",
    schema_name="ML"
)

print("✅ Connexion au Model Registry établie")
print(f"   Database : HOUSE_PRICE_DB")
print(f"   Schema   : ML")

In [ ]:
# Enregistrement du meilleur modèle 
# On enregistre le modèle avec ses métriques et métadonnées pour assurer la traçabilité et la reproductibilité.

# Métriques du modèle champion
champion_metrics = {
    "r2"   : float(final_df.iloc[0]['R²']),
    "rmse" : float(final_df.iloc[0]['RMSE']),
    "mae"  : float(final_df.iloc[0]['MAE']),
}

# Enregistrement dans le registry
model_version = reg.log_model(
    model=best_final_model,
    model_name="HOUSE_PRICE_PREDICTOR",
    version_name="V1",
    comment=(
        f"Modèle champion : {best_final_name}. "
        f"Entraîné sur {X_train.shape[0]} exemples. "
        f"R²={champion_metrics['r2']:.4f} | "
        f"RMSE={champion_metrics['rmse']:,.0f} | "
        f"MAE={champion_metrics['mae']:,.0f}"
    ),
    metrics=champion_metrics,
    conda_dependencies=["scikit-learn", "xgboost"],
    sample_input_data=X_train.head(10),
)

print("✅ Modèle enregistré avec succès dans le Registry !")
print(f"   Nom      : HOUSE_PRICE_PREDICTOR")
print(f"   Version  : V1")
print(f"   R²       : {champion_metrics['r2']:.4f}")
print(f"   RMSE     : {champion_metrics['rmse']:,.0f}")
print(f"   MAE      : {champion_metrics['mae']:,.0f}")

In [ ]:
# Vérification et listing des modèles enregistrés
# On confirme que le modèle est bien présent dans le registry et qu'il est prêt pour être utilisé en production.

# Lister tous les modèles disponibles
models_in_registry = reg.show_models()
print("=== Modèles disponibles dans le Registry ===")
display(models_in_registry)

# Charger le modèle pour vérifier qu'il est bien accessible
loaded_model = reg.get_model("HOUSE_PRICE_PREDICTOR").version("V1")

# Test rapide de chargement
test_pred = loaded_model.run(X_test.head(3), function_name="predict")
print("\n✅ Modèle chargé et opérationnel depuis le Registry")
print(f"   Test prédictions (3 premières lignes) : {test_pred}")
print(f"   Valeurs réelles                       : {y_test.head(3).values}")

## Etape 9 — Inférence

In [ ]:
# Inférence avec le modèle du Model Registry
# On charge le modèle directement depuis le Snowflake Model Registry pour générer des prédictions sur de nouvelles données, sans jamais exporter le modèle hors de Snowflake.

from snowflake.ml.registry import Registry

# Connexion au registry
reg = Registry(
    session=session,
    database_name="HOUSE_PRICE_DB",
    schema_name="ML"
)

# Chargement du modèle champion enregistré à l'étape 8
loaded_model = reg.get_model("HOUSE_PRICE_PREDICTOR").version("V1")

print("✅ Modèle HOUSE_PRICE_PREDICTOR V1 chargé depuis le Registry")

In [ ]:
# Test d'inférence sur une maison fictive
# On simule une nouvelle maison dont on veut estimer le prix comme le ferait un utilisateur métier en production.

# Nouvelle maison à évaluer
nouvelle_maison = pd.DataFrame([{
    'AREA'            : 5000,   # 5000 m²
    'BEDROOMS'        : 4,      # 4 chambres
    'BATHROOMS'       : 2,      # 2 salles de bain
    'STORIES'         : 2,      # 2 étages
    'MAINROAD'        : 1,      # accès route principale
    'GUESTROOM'       : 1,      # chambre d'amis
    'BASEMENT'        : 0,      # pas de sous-sol
    'HOTWATERHEATING' : 0,      # pas de chauffage eau chaude
    'AIRCONDITIONING' : 1,      # climatisation
    'PARKING'         : 2,      # 2 places de parking
    'PREFAREA'        : 1,      # zone privilégiée
    'FURNISHINGSTATUS': 2       # meublée
}])

# Appliquer le même scaler que lors de l'entraînement
nouvelle_maison_scaled = nouvelle_maison.copy()
nouvelle_maison_scaled[num_cols] = scaler.transform(nouvelle_maison[num_cols])

# Inférence — retourne un DataFrame Snowflake ML
prediction = loaded_model.run(nouvelle_maison_scaled, function_name="predict")

# Extraction correcte : iloc[0,0] = première ligne, première colonne
prix_estime = float(prediction.iloc[0, 0])

print("🏠 Caractéristiques de la maison :")
print(f"   Surface        : 5000 m²")
print(f"   Chambres       : 4  |  Salles de bain : 2")
print(f"   Étages         : 2  |  Parking        : 2 places")
print(f"   Climatisation  : Oui | Zone privilégiée : Oui")
print(f"   Ameublement    : Meublée")
print(f"\n💰 Prix estimé : {prix_estime:,.0f}")

In [ ]:
# Inférence en masse sur le jeu de test
# On applique le modèle sur l'ensemble des données de test et on stocke les résultats dans une table Snowflake pour traçabilité et analyse ultérieure.

# Prédictions sur tout le jeu de test
X_test_reset    = X_test.reset_index(drop=True)
y_test_reset    = y_test.reset_index(drop=True)
predictions_all = loaded_model.run(X_test_reset, function_name="predict")

# Construction du DataFrame de résultats
results_inference = X_test_reset.copy()
results_inference['PRIX_REEL']   = y_test_reset
results_inference['PRIX_PREDIT'] = predictions_all
results_inference['ECART_ABS']   = abs(
    results_inference['PRIX_REEL'] - results_inference['PRIX_PREDIT']
)
results_inference['ECART_PCT']   = (
    results_inference['ECART_ABS'] / results_inference['PRIX_REEL'] * 100
).round(2)

# Aperçu des résultats
print("=== Aperçu des prédictions ===")
display(results_inference[['PRIX_REEL','PRIX_PREDIT','ECART_ABS','ECART_PCT']].head(10))

print(f"\n📊 Résumé des performances en inférence :")
print(f"   Nb prédictions          : {len(results_inference)}")
print(f"   Écart moyen             : {results_inference['ECART_ABS'].mean():,.0f}")
print(f"   Écart moyen (%)         : {results_inference['ECART_PCT'].mean():.2f}%")
print(f"   Dans ±10% du prix réel  : {(results_inference['ECART_PCT'] < 10).mean()*100:.1f}%")
print(f"   Dans ±20% du prix réel  : {(results_inference['ECART_PCT'] < 20).mean()*100:.1f}%")

In [ ]:
# Stockage des prédictions dans Snowflake
# Les résultats d'inférence sont persistés dans une table dédiée pour permettre le suivi des performances du modèle en production (model monitoring).

# Sauvegarde dans Snowflake
session.create_dataframe(results_inference) \
       .write.mode("overwrite") \
       .save_as_table("HOUSE_PRICES_PREDICTIONS")

# Vérification depuis Snowflake
count = session.table("HOUSE_PRICES_PREDICTIONS").count()
print(f"✅ {count} prédictions stockées dans la table HOUSE_PRICES_PREDICTIONS")

# Vérification SQL
print("\n=== Aperçu depuis Snowflake ===")
session.sql("""
    SELECT
        PRIX_REEL,
        PRIX_PREDIT,
        ECART_PCT,
        CASE
            WHEN ECART_PCT < 10 THEN '✅ Excellent (< 10%)'
            WHEN ECART_PCT < 20 THEN '🟡 Bon (< 20%)'
            ELSE '🔴 À améliorer (> 20%)'
        END AS QUALITE_PREDICTION
    FROM HOUSE_PRICES_PREDICTIONS
    LIMIT 10
""").show()